In [5]:
# !pip install gspread
# pip install oauth2client
import gspread
print("gspread is working!")

ModuleNotFoundError: No module named 'gspread'

In [2]:
python3.10 -m venv spark_env

SyntaxError: invalid syntax (3184123163.py, line 1)

In [1]:
import importlib
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    try:
        importlib.import_module(import_name or package_name)
        print(f"✅ '{package_name}' is already installed.")
    except ImportError:
        print(f"📦 Installing '{package_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"✅ '{package_name}' installed successfully.")

# Example usage
install_if_missing("pyspark")
install_if_missing("gspread")
install_if_missing("pandas")

✅ 'pyspark' is already installed.
✅ 'gspread' is already installed.
📦 Installing 'pandas'...
✅ 'pandas' installed successfully.


In [2]:
# !pip install pyspark
import pyspark
print("PySpark is working!")

PySpark is working!


In [ ]:
#Spark Connection
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkAnalytics").getOrCreate()
print("Spark is working!")

: 

In [ ]:
import urllib.request

# This script fetches data from a specified URL and prints the content.
# Download the all sheets information locally
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQO7atdyPm1anKXSql2oz74C3To18tkzYRIPqhm9YqWX1w3nm73sL8lpybaykBO2g7IB7cvIpQ8W9_u/pub?gid=0&single=true&output=csv"
allsheetsInformation = "AllSheets_Link.csv"
urllib.request.urlretrieve(url, allsheetsInformation)
print("Data downloaded successfully!")

# --- IGNORE ---
with open(allsheetsInformation, 'r') as file:
    content = file.read()
    print(content)


In [ ]:
import csv
with open(allsheetsInformation, 'r') as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

In [ ]:
metadata_path = "AllSheets_Link.csv"
metadata_df = spark.read \
    .option("header", True)\
    .option("nullValue", "NULL") \
    .csv(metadata_path)
# metadata_df.show()

sheet_info = metadata_df.select("Table_name", "Link").collect()

# print(sheet_info)

dataframes = {}

for row in sheet_info:
    df_name = row["Table_name"]
    csv_url = row["Link"]
    local_path = f"/tmp/{df_name}.csv" 
    urllib.request.urlretrieve(csv_url, local_path)
    df = spark.read \
        .option("header", True) \
        .option("nullValue", "NULL") \
        .option("inferSchema", True) \
        .csv(local_path)
    dataframes[df_name] = df

# print(dataframes.keys())




|DepartmentName          |NationalIDNumber|JobTitle                         |
|---|---|---|
|Executive               |295847284       |Chief Executive Officer          |
|Engineering             |245797967       |Vice President of Engineering    |
|Engineering             |509647174       |Engineering Manager              |
|Engineering             |112457891       |Senior Tool Designer             |
|Tool Design             |112457891       |Senior Tool Designer             |
|Engineering             |695256908       |Design Engineer                  |
|Engineering             |998320692       |Design Engineer                  |
|Research and Development|134969118       |Research and Development Manager |
|Research and Development|811994146       |Research and Development Engineer|
|Research and Development|658797903       |Research and Development Engineer|


In [ ]:
from pyspark.sql.functions import col, when, count, isnan, sum
d=dataframes.get("Department").alias("d")
e=dataframes.get("Employee").alias("e")
ed=dataframes.get("EmployeeDepartmentHistory").alias("ed")

# e.printSchema()
# ed.printSchema()
# d.printSchema()

joined_df = ed.join(d, d["DepartmentID"] == ed["DepartmentID"], how="inner") \
            .join(e, e["BusinessEntityID"] == ed["BusinessEntityID"], how="inner") \
            .select(col("d.Name").alias("DepartmentName"), "e.NationalIDNumber", "e.JobTitle")

joined_df.show(10, truncate=False)

joined_df = None

``` sql
SELECT soh.SalesOrderNumber,
    soh.OrderDate, soh.Status, soh.OnlineOrderFlag, soh.CustomerID, soh.SalesPersonID, st.Name AS TerritoryName,
    st.CountryRegionCode AS TerritoryRegion, soh.SubTotal, soh.TaxAmt, soh.Freight, soh.TotalDue,
    sod.SpecialOfferID, sod.ProductID, sod.OrderQty, sod.UnitPrice, sod.UnitPriceDiscount, sod.LineTotal, sod.CarrierTrackingNumber
FROM Sales.SalesOrderHeader AS soh INNER JOIN
    Sales.SalesOrderDetail AS sod
    ON  soh.SalesOrderID = sod.SalesOrderID INNER JOIN
    Sales.SalesTerritory AS st
    ON  soh.TerritoryID = st.TerritoryID
WHERE   SalesOrderNumber = 'SO43659'
```

In [ ]:
from pyspark.sql.functions import col, when, count, isnan, sum
soh = dataframes.get("SalesOrderHeader").alias("soh")
sod = dataframes.get("SalesOrderDetail").alias("sod")
st = dataframes.get("SalesTerritory").alias("st")

joined_df = soh.join(sod, soh["SalesOrderID"] == sod["SalesOrderID"], how="inner") \
            .join(st, st["TerritoryID"] == soh["TerritoryID"], how="inner") \
            .select(col("soh.SalesOrderNumber"), 
                    col("soh.OrderDate"), 
                    col("soh.Status"), 
                    col("soh.OnlineOrderFlag"), 
                    col("soh.CustomerID"), 
                    col("soh.SalesPersonID"), 
                    col("st.Name").alias("TerritoryName"),
                    col("st.CountryRegionCode").alias("TerritoryRegion"),
                    col("soh.SubTotal"),
                    col("soh.TaxAmt"),
                    col("soh.Freight"),
                    col("soh.TotalDue"),
                    col("sod.SpecialOfferID"),
                    col("sod.ProductID" ),
                    col("sod.OrderQty"),
                    col("sod.UnitPrice"),
                    col("sod.UnitPriceDiscount"),
                    col("sod.LineTotal"),
                    col("sod.CarrierTrackingNumber")
                    )
joined_df.filter(col("SalesOrderNumber") == "SO43659").show(truncate=False)


### Slice data based on Territory
#### Year 2014 and 2013 compared

In [ ]:
from pyspark.sql import functions as F

# Slice data based on Territory
# Step 1: Collect required columns:
step_1_df = joined_df.select(
    F.year(F.to_date(F.col("OrderDate"), "M/d/yyyy")).alias("OrderYear"),
    F.month(F.to_date(F.col("OrderDate"), "M/d/yyyy")).alias("OrderMonth"),
    "TerritoryName",
    "TerritoryRegion",
    "SubTotal",
    "TaxAmt",
    "Freight",
    "TotalDue"
)

# Step 2: aggregated majors based on Year, Month, TerritoryName, TerritoryRegion
step_2_df = step_1_df.groupBy("OrderYear","OrderMonth","TerritoryName","TerritoryRegion") \
    .agg(F.round(sum("SubTotal"), 2).alias("SubTotal"),
         F.round(sum("TaxAmt"), 2).alias("TaxAmt"), 
         F.round(sum("Freight"), 2).alias("Freight"), 
         F.round(sum("TotalDue"), 2).alias("TotalDue")
         ) \
    .sort(F.desc("OrderYear"),F.desc("OrderMonth"),F.desc("TerritoryName"),F.desc("TerritoryRegion"))



# Step 3: calculate YTD, and Last Year Data

# Get current YTD cutoff (e.g., June)
current_ytd_month = 6  # or dynamically: datetime.now().month


# Filter for 2014 and 2013 YTD
filtered_df_year = step_2_df.filter(
    ((F.col("OrderYear") == 2014) | (F.col("OrderYear") == 2013)) &
    (F.col("OrderMonth") <= current_ytd_month)
)

# filtered_df_year.show(100, truncate=False)

# Aggregate by year and territory
agg_df_year = filtered_df_year.groupBy("OrderYear", "TerritoryName", "TerritoryRegion") \
    .agg(
        F.round(F.sum("SubTotal"), 2).alias("SubTotal"),
        F.round(F.sum("TaxAmt"), 2).alias("TaxAmt"),
        F.round(F.sum("Freight"), 2).alias("Freight"),
        F.round(F.sum("TotalDue"), 2).alias("TotalDue")
    ) \
    .orderBy("OrderYear", ascending=False)
# agg_df_year.show(truncate=False)

# Step 4: Pivot for side-by-side comparison
pivot_df_year = agg_df_year.groupBy("TerritoryName", "TerritoryRegion") \
    .pivot("OrderYear", [2014, 2013]) \
    .agg(
        F.first("SubTotal").alias("SubTotal"),
        F.first("TaxAmt").alias("TaxAmt"),
        F.first("Freight").alias("Freight"),
        F.first("TotalDue").alias("TotalDue")
    ) \
    .orderBy("TerritoryName", "TerritoryRegion")
# pivot_df_year.show(truncate=False)

final_df_year = pivot_df_year.select(
    "TerritoryName", "TerritoryRegion",
    F.col("2014_SubTotal").alias("SubTotal_2014"),
    F.col("2013_SubTotal").alias("SubTotal_2013"),
    F.col("2014_TaxAmt").alias("TaxAmt_2014"),
    F.col("2013_TaxAmt").alias("TaxAmt_2013"),
    F.col("2014_Freight").alias("Freight_2014"),
    F.col("2013_Freight").alias("Freight_2013"),
    F.col("2014_TotalDue").alias("TotalDue_2014"),
    F.col("2013_TotalDue").alias("TotalDue_2013")
)

final_df_year.show(100, truncate=False)

### Slice data based on Territory
#### Month May, 2014 and May, 2013 compared

In [ ]:
current_ytd_month = 5  # or dynamically: datetime.now().month

# Filter for 2014 and 2013 YTD
filtered_df_May = step_2_df \
    .filter((F.col("OrderMonth") == current_ytd_month) & ((F.col("OrderYear") == 2014) | (F.col("OrderYear") == 2013))) \
    .select(
        "OrderYear",
        "TerritoryName",
        "TerritoryRegion",
        "SubTotal",
        "TaxAmt",
        "Freight",
        "TotalDue"
    )

filtered_df_May_agg = filtered_df_May.groupBy("OrderYear", "TerritoryName", "TerritoryRegion") \
    .agg(sum("SubTotal").alias("SubTotal"),
         sum("TaxAmt").alias("TaxAmt"),
            sum("Freight").alias("Freight"),
            sum("TotalDue").alias("TotalDue")
            ) \
    .sort(F.desc("OrderYear"),F.desc("TerritoryName"),F.desc("TerritoryRegion"))

filtered_df_May_agg_pivot = filtered_df_May_agg.groupBy("TerritoryName", "TerritoryRegion") \
    .pivot("OrderYear", [2014, 2013]) \
    .agg(
        F.first("SubTotal").alias("SubTotal"),
        F.first("TaxAmt").alias("TaxAmt"),
        F.first("Freight").alias("Freight"),
        F.first("TotalDue").alias("TotalDue")
    ) \
    .orderBy("TerritoryName", "TerritoryRegion")

# filtered_df_May_agg_pivot.show(100, truncate=False)

final_df_May = filtered_df_May_agg_pivot.select(
    "TerritoryName", "TerritoryRegion",
    F.col("2014_SubTotal").alias("SubTotal_2014_May"),
    F.col("2013_SubTotal").alias("SubTotal_2013_May"),
    F.col("2014_TaxAmt").alias("TaxAmt_2014_May"),
    F.col("2013_TaxAmt").alias("TaxAmt_2013_May"),
    F.col("2014_Freight").alias("Freight_2014_May"),
    F.col("2013_Freight").alias("Freight_2013_May"),
    F.col("2014_TotalDue").alias("TotalDue_2014_May"),
    F.col("2013_TotalDue").alias("TotalDue_2013_May")
)

final_df_May.show(100, truncate=False)





### Join year and Month

In [ ]:
fy = final_df_year.alias("fy")
fm = final_df_May.alias("fm")

final_df_year_month = fy.join(fm,
    (fm["TerritoryRegion"] == fy["TerritoryRegion"]) & (fm["TerritoryName"] == fy["TerritoryName"]), how="inner")

final_df_year_month \
.select(
    "fy.TerritoryRegion",
    "fy.TerritoryName",
    "fy.SubTotal_2014",
    "fy.SubTotal_2013",
    "fm.SubTotal_2014_May",
    "fm.SubTotal_2013_May",
    "fy.TaxAmt_2014",
    "fy.TaxAmt_2013",
    "fm.TaxAmt_2014_May",
    "fm.TaxAmt_2013_May",
    "fy.Freight_2014",
    "fy.Freight_2013",
    "fm.Freight_2014_May",
    "fm.Freight_2013_May",
    "fy.TotalDue_2014",
    "fy.TotalDue_2013",
    "fm.TotalDue_2014_May",
    "fm.TotalDue_2013_May"
    ) \
.sort("TerritoryRegion", "TerritoryName") \
.show(100, truncate=False)

In [ ]:
# joined_df.show(10, truncate=False)

# Calculate sale on USA based on customer for year 2014 and 2013

from pyspark.sql import functions as F
step_1_df = joined_df \
    .filter(F.col("TerritoryRegion") == "US") \
    .select(
    F.year(F.to_date(F.col("OrderDate"), "M/d/yyyy")).alias("OrderYear"),
    "customerID",
    "SubTotal",
    "TaxAmt",
    "Freight",
    "TotalDue"
    )
    
step_1_df_agg = step_1_df.groupBy("OrderYear","customerID") \
    .agg(F.round(sum("SubTotal"), 2).alias("SubTotal"),
         F.round(sum("TaxAmt"), 2).alias("TaxAmt"), 
         F.round(sum("Freight"), 2).alias("Freight"), 
         F.round(sum("TotalDue"), 2).alias("TotalDue")
         ) \
    .filter(F.col("OrderYear").isin([2013,2014])) \
    .sort(F.desc("OrderYear"),F.desc("customerID"))

step_1_df_agg_pivot = step_1_df_agg.groupBy("customerID") \
    .pivot("OrderYear", [2014, 2013]) \
    .agg(
        F.first("SubTotal").alias("SubTotal"),
        F.first("TaxAmt").alias("TaxAmt"),
        F.first("Freight").alias("Freight"),
        F.first("TotalDue").alias("TotalDue")
    ) \
    .orderBy("customerID")

step_1_df_agg_pivot.show(100, truncate=False)



|TerritoryName |TerritoryRegion|TotalSubTotal|TotalTaxAmt|TotalFreight|TotalDueAmount|
|--------------|---------------|-------------|-----------|------------|--------------|
|Australia     |AU             |2.259747288E7|2075380.67 |648556.6    |2.532141017E7 |
|Canada        |CA             |6.060220383E7|5828318.59 |1821349.64  |6.825187207E7 |
|Central       |US             |2.790692117E7|2710736.79 |847105.25   |3.146476321E7 |
|France        |FR             |3.080088759E7|2950522.17 |922038.23   |3.467344799E7 |
|Germany       |DE             |2.445433616E7|2311812.28 |722441.41   |2.748858984E7 |
|Northeast     |US             |2.487495209E7|2391747.04 |747420.95   |2.801412008E7 |
|Northwest     |US             |6.169001924E7|5953680.37 |1860525.22  |6.950422485E7 |
|Southeast     |US             |2.422258467E7|2326746.72 |727108.34   |2.727643975E7 |
|Southwest     |US             |7.408366915E7|7105776.66 |2220555.35  |8.341000117E7 |
|United Kingdom|GB             |4.002215411E7|3819992.48 |1193747.71  |4.503589431E7 |

In [ ]:
import pyodbc

conn = pyodbc.connect("Driver={};"
"Server="
"Trusted_Connection=yes;")
cur_hist = conn.cursor()